In [ ]:
!pip install sentence-transformers

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load the embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dim = embedding_model.get_sentence_embedding_dimension()

In [ ]:
!pip install faiss-cpu

In [16]:
import torch

In [17]:
class RAG:
    def __init__(self, embedding_model, embedding_dim):
        self.embedding_model = embedding_model
        self.embedding_dim = embedding_dim
        self.faiss_index = faiss.IndexFlatIP(embedding_dim)  # Use inner product for cosine
        self.context_keys = []
        self.contexts = {}

    def index(self, context_key, context):
        print("/// [RAG] indexing context: " + context_key)
        embeddings_np = self.embedding_model.encode([context], normalize_embeddings=True)
        self.faiss_index.add(embeddings_np.astype(np.float32))
        self.context_keys.append(context_key)
        self.contexts[context_key] = context

    def search(self, question, top_k=5):
        print("/// [RAG] searching for question: " + question)
        question_np = self.embedding_model.encode([question], normalize_embeddings=True)
        distances, ids = self.faiss_index.search(question_np.astype(np.float32), top_k)
        print("//// Distances: ", distances)
        print("//// IDs: ", ids)
        context_key = self.context_keys[ids[0][0]]
        context = self.contexts[context_key]
        return context_key, context

In [18]:
# import torch
# import faiss
# import numpy as np

# class RAG:
#     def __init__(self, model, tokenizer, device, embedding_dim):
#         self.model = model
#         self.tokenizer = tokenizer
#         self.device = device
#         self.embedding_dim = embedding_dim
#         self.faiss_index = faiss.IndexFlatIP(embedding_dim)  # Use inner product for cosine
#         self.context_keys = []
#         self.contexts = {}

#     def index(self, context_key, context):
#         print("/// [RAG] indexing context: " + context_key)
#         embeddings = self._get_llm_embeddings(context)
#         # Normalize embedding for cosine similarity
#         embeddings_np = embeddings.cpu().numpy().reshape(1, -1).astype(np.float32)
#         embeddings_np /= np.linalg.norm(embeddings_np, axis=1, keepdims=True)
#         self.faiss_index.add(embeddings_np)
#         self.context_keys.append(context_key)
#         self.contexts[context_key] = context

#     def search(self, question, top_k=5):
#         print("/// [RAG] searching for question: " + question)
#         question_embedding = self._get_llm_embeddings(question)
#         question_np = question_embedding.cpu().numpy().reshape(1, -1).astype(np.float32)
#         question_np /= np.linalg.norm(question_np, axis=1, keepdims=True)
#         distances, ids = self.faiss_index.search(question_np, top_k)
#         print("//// Distances: ", distances)
#         print("//// IDs: ", ids)
#         context_key = self.context_keys[ids[0][0]]
#         context = self.contexts[context_key]
#         return context_key, context

#     def _get_llm_embeddings(self, text):
#         inputs = self.tokenizer(
#             text, return_tensors="pt", truncation=True, padding=True
#         )
#         with torch.no_grad():
#             outputs = self.model(**inputs)
#         last_hidden_state = outputs.last_hidden_state
#         embeddings = last_hidden_state.mean(dim=1).squeeze()
#         return embeddings

In [19]:
# import torch
# import faiss
# import numpy as np


# class RAG:

#     def __init__(self, model, tokenizer, device, embedding_dim=768):
#         self.model = model
#         self.tokenizer = tokenizer
#         self.device = device
#         self.embedding_dim = embedding_dim
#         self.faiss_index = faiss.IndexFlatL2(embedding_dim)  # create faiss index
#         self.context_keys = []
#         self.contexts = {}

#     def index(self, context_key, context):
#         print("/// [RAG] indexing context: " + context_key)
#         embeddings = self._get_llm_embeddings(context)
#         # print(
#         #     "/// -> [index] Embedding dimension: ",
#         #     len(embeddings),
#         #     " and shape: ",
#         #     embeddings.shape,
#         # )
#         embeddings_np = embeddings.cpu().numpy().reshape(1, -1).astype(np.float32)
#         # print(
#         #     "/// -> [index] Embedding (reshape) dimension: ",
#         #     len(embeddings_np),
#         #     " and shape: ",
#         #     embeddings_np.shape,
#         # )
        
#         self.faiss_index.add(embeddings_np)
#         self.context_keys.append(context_key)
#         self.contexts[context_key] = context

#     def search(self, question, top_k=5):
#         print("/// [RAG] searching for question: " + question)
#         question_embedding = self._get_llm_embeddings(question)
#         # print(
#         #     "/// -> [search] embedding dimension: ",
#         #     len(question_embedding),
#         #     " and shape: ",
#         #     question_embedding.shape,
#         # )
#         question_np = question_embedding.cpu().numpy().reshape(1, -1).astype(np.float32)
#         # print(
#         #     "/// -> [search] embedding (reshape) dimension: ",
#         #     len(question_np),
#         #     " and shape: ",
#         #     question_np.shape,
#         # )
#         distances, ids = self.faiss_index.search(question_np, top_k)
#         print("//// Distances: ", distances)
#         print("//// IDs: ", ids)
#         context_key = self.context_keys[ids[0][0]]
#         context = self.contexts[context_key]
#         return context_key, context

#     def _get_llm_embeddings(self, text):
#         # Tokenize the text chunk
#         inputs = self.tokenizer(
#             text, return_tensors="pt", truncation=True, padding=True
#         )

#         # Generate embeddings using the model
#         with torch.no_grad():
#             outputs = self.model(**inputs)

#         # Extract the last hidden state (token-level embeddings)
#         last_hidden_state = outputs.last_hidden_state
#         # Pool the token embeddings to get a single vector (mean pooling)
#         embeddings = last_hidden_state.mean(dim=1).squeeze()

#         # print(f"[get_emb] Last hidden state shape: {outputs.last_hidden_state.shape}")
#         # print(f"[get_emb] Model config hidden size: {self.model.config.hidden_size}")
#         # print(f"[get_emb] Embeddings length: {len(embeddings)}")
#         # print(f"[get_emb] Embeddings shape: {embeddings.shape}")
#         return embeddings

#     def _get_llm_embeddings_2(self, text):
#         # Tokenize input text
#         inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
#         with torch.no_grad():
#             # Forward pass through the model to get hidden states
#             outputs = self.model(**inputs, output_hidden_states=True)
#             # Extract the last hidden state (embeddings)
#             hidden_states = outputs.hidden_states
#             last_hidden_state = hidden_states[-1]
#             # Mean pooling for sentence-level embedding
#             embeddings = last_hidden_state.mean(dim=1)
#             # print(f"[get_emb_2] Last hidden state shape: {outputs.last_hidden_state.shape}")
#             # print(f"[get_emb_2] Model config hidden size: {self.model.config.hidden_size}")
#             # print(f"[get_emb_2] Embeddings length: {len(embeddings)}")
#             # print(f"[get_emb_2] Embeddings shape: {embeddings.shape}")
#         return embeddings


In [20]:
# rag_instance = RAG(
#     model=model, tokenizer=tokenizer, device="cpu", embedding_dim=1024
# )
# rag_instance = RAG(
#     model=model, tokenizer=tokenizer, device="cpu", embedding_dim=embedding_dim
# )
rag_instance = RAG(embedding_model=embedding_model, embedding_dim=embedding_dim)

In [21]:
text = "Encode this text please, into an embedding"

In [ ]:
text_emb = rag_instance._get_llm_embeddings(text)

In [ ]:
text_emb_2 = rag_instance._get_llm_embeddings_2(text)

In [22]:
rag_instance.index(context_key="contextkey", context=text)

/// [RAG] indexing context: contextkey


In [23]:
rag_instance.search(question=text)

/// [RAG] searching for question: Encode this text please, into an embedding
//// Distances:  [[ 1.0000000e+00 -3.4028235e+38 -3.4028235e+38 -3.4028235e+38
  -3.4028235e+38]]
//// IDs:  [[ 0 -1 -1 -1 -1]]


('contextkey', 'Encode this text please, into an embedding')

In [24]:
import os

data_dir = "../frontend/data/"
for filename in os.listdir(data_dir):
    if filename.endswith(".txt"):
        file_path = os.path.join(data_dir, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
            rag_instance.index(context_key=filename, context=content)

/// [RAG] indexing context: extra-compound-mape.txt
/// [RAG] indexing context: extra-biology.txt
/// [RAG] indexing context: extra-estimate-performance.txt
/// [RAG] indexing context: extra-leonardo.txt
/// [RAG] indexing context: SplitRPC-sigmetrics23.txt
/// [RAG] indexing context: extra-finance.txt
/// [RAG] indexing context: aplomb-sigcomm12.txt
/// [RAG] indexing context: vllm.txt
/// [RAG] indexing context: click.txt
/// [RAG] indexing context: extra-music.txt
/// [RAG] indexing context: nsdi20-paper-barbette.txt
/// [RAG] indexing context: sigcomm24-crux.txt
/// [RAG] indexing context: extra-oss-monitoring.txt
/// [RAG] indexing context: metron-nsdi18.txt
/// [RAG] indexing context: extra-distributed-monitoring.txt


In [ ]:
import os

prompts_dir = "../frontend/prompts/"
accuracy = 0
prompt_files = os.listdir(prompts_dir)
for prompt_filename in prompt_files:
    if prompt_filename.endswith(".txt"):
        prompt_path = os.path.join(prompts_dir, prompt_filename)
        with open(prompt_path, "r", encoding="utf-8") as prompt_file:
            first_line = prompt_file.readline().strip()
            context_key, _ = rag_instance.search(question=first_line)
            if prompt_filename == context_key:
                accuracy += 1
            print(f"////// Matched: {prompt_filename == context_key}")

print(f"# Accuracy: {(accuracy / len(prompt_files)) * 100}%")

/// [RAG] searching for question: What is the main motivation for using decentralized control in self-adaptive systems according to the paper?
//// Distances:  [[0.6217058  0.21929428 0.19829683 0.18997075 0.1785673 ]]
//// IDs:  [[ 1 11 14  7  5]]
////// Matched: True
/// [RAG] searching for question: What is the definition of epigenetics?
//// Distances:  [[0.6104646  0.0752614  0.07387325 0.04357736 0.03292998]]
//// IDs:  [[2 0 1 8 9]]
////// Matched: True
/// [RAG] searching for question: What is the main problem addressed by the PAPE algorithm in the context of machine learning model deployment?
//// Distances:  [[0.52827495 0.3198614  0.22368547 0.2180513  0.20323795]]
//// IDs:  [[ 3  8 12  5 15]]
////// Matched: True
/// [RAG] searching for question: Who is described as the quintessential Renaissance man in the report?
//// Distances:  [[0.49107942 0.13006036 0.05337492 0.04378686 0.04364811]]
//// IDs:  [[ 4  0  6 12 10]]
////// Matched: True
/// [RAG] searching for question: